In [4]:
import importlib
import sys
import os
import argparse

def Versions():
        def PrintVer(libname, version):
                print(f"{str(libname) + ' version:':28s} {version}")

        def TryPrintVersion(libname):
                try:
                        lib = importlib.import_module(libname)
                        try:
                                v = lib.__version__
                                PrintVer(libname, lib.__version__)
                        except:
                                print(f"WARNING: library '{libname}' does not have a '__version__' attribute", file=sys.stderr)
                except ModuleNotFoundError:
                        print(f"WARNING: could not find library '{libname}' in path", file=sys.stderr)

        PrintVer("Python", f"{sys.version_info.major}.{sys.version_info.minor}.{sys.version_info.micro}")

        TryPrintVersion("numpy")
        TryPrintVersion("sklearn")
        TryPrintVersion("keras")
        TryPrintVersion("tensorflow")
        TryPrintVersion("tensorflow.keras")
        TryPrintVersion("cv2")
        TryPrintVersion("torch")
        TryPrintVersion("libitmal")


def TestGPU(gpunumber, verbose=False):
        try:
                if verbose:
                        print("UseGPU()..")

                import tensorflow

                #n = str(gpunumber)
                #tf_device=f"/gpu:{gpunumber}"

                key = "CUDA_VISIBLE_DEVICES"
                CUDA_VISIBLE_DEVICES = os.environ[key] if key in os.environ else None
                gpus_all_physical_list = tensorflow.config.list_physical_devices(device_type='GPU')    

                print(f"   CUDA_VISIBLE_DEVICES   = {CUDA_VISIBLE_DEVICES}")
                print(f"   gpus_all_physical_list = ")
                for i in gpus_all_physical_list:
                        print(f"        {i}")
        except Exception as e:
                if verbose:
                        WARNMSG(f"ERROR: something failed in UseGPU(), re-raising exception='{e}'\n")
                raise e


def DisableTFWarns():
        key = "TF_CPP_MIN_LOG_LEVEL"
        os.environ[key] = '3' # to disable TF warnings


def TestKeras():
        import numpy as np
        from libitmal import dataloaders 

        X, y = dataloaders.MNIST_GetDataSet()

        X = X.reshape(70000, 784)
        #X_norm = X*np.float32(1)  # NOTE: ups, remembered convert to float but forgot scale 
        X_norm = X/np.float32(255) # NOTE: remembered convert to float and scale 

        print(f"X_norm.shape={X_norm.shape}")
        print(f"  type(X_norm[0][0])={type(X_norm[0][0])}")
        print(f"  X_norm.dtype={X_norm.dtype}")
        print(f"  np.max(X_norm)={np.max(X_norm)}")
        print(f"  np.min(X_norm)={np.min(X_norm)}")

        ###########################################################################

        from sklearn.model_selection import train_test_split
        from keras.utils import to_categorical

        X_train, X_test, y_train, y_test = train_test_split(X_norm, y, test_size=0.3, random_state=42)

        y_train_cat = to_categorical(y_train)
        y_test_cat  = to_categorical(y_test)

        ###############################################################################

        from time import time

        from tensorflow.keras.models import Sequential
        from tensorflow.keras.layers import Dense, Dropout
        from tensorflow.keras.optimizers import Adam, SGD, Nadam
        #from tensorflow.keras.layers.normalization import BatchNormalization
        from tensorflow.keras.backend import batch_normalization

        def AddLayer(model, units, add_normalization=False, dropout_rate=-1):
                model.add(Dense(units=units, 
                                        activation="elu", 
                                        kernel_initializer="he_normal",  
                                        bias_initializer  ="he_normal"))     
                if add_normalization:
                        model.add(BatchNormalization())
                if dropout_rate>=0:
                        model.add(Dropout(rate=dropout_rate))

        model = Sequential()
        #model.add(BatchNormalization())
        model.add(Dense(input_dim=(784), units=20, 
                                        activation="elu", 
                                        kernel_initializer="he_normal",  
                                        bias_initializer  ="he_normal"))     
        #model.add(BatchNormalization())
        #model.add(Dropout(rate=DROPOUT_RATE))

        AddLayer(model, 50)
        AddLayer(model, 70)
        AddLayer(model, 100)
        AddLayer(model, 70)
        AddLayer(model, 50)
        AddLayer(model, 20)

        model.add(Dense(units=10, 
                                        activation="softmax",
                                        kernel_initializer="he_normal",  
                                        bias_initializer  ="he_normal"))     

        def ModelCompile(model):
                #optimizer = Adam (learning_rate=0.1) # NOTE: will fail miserable for learning_rate=0.1
                optimizer = Adam (learning_rate=0.01) 
                #optimizer = Nadam(learning_rate=0.002) # NOTE: Nadam=Nesterov Adam optimizer.
                #optimizer = SGD(learning_rate=0.01, decay=1e-6, momentum=0.9, nesterov=True)
                          
                model.compile(loss='categorical_crossentropy', 
                                          optimizer=optimizer, 
                                           metrics=['categorical_crossentropy',
                                                                'categorical_accuracy',
                                                                'mean_squared_error',
                                                                'mean_absolute_error'])
                try:
                        model.summary()
                except:
                        print("WARNING: model.summary() failed, cannot print models with BatchNormalization() layers before fit")


        def PrintScore(model, X_test, y_test_cat, t):
                #print(history.history)
                score = model.evaluate(X_test, y_test_cat, verbose=0)
                print(f"Training time: {t:0.1f} sec")
                print(f"Test loss:     {score[0]}") # loss is score 0 by definition?
                print(f"Test accuracy: {score[2]}")
                #print(f"All scores in history: {score}")

        # Train
        VERBOSE     = 1
        EPOCHS      = 1
        DROPOUT_RATE= 0.1

        ModelCompile(model)

        start = time()
        history = model.fit(X_train, y_train_cat, 
                                                validation_data=(X_test, y_test_cat), 
                                                epochs=EPOCHS, 
                                                verbose=VERBOSE)
        t = time()-start

        print("")
        PrintScore(model, X_test, y_test_cat, t)
        print("\nOK")

def TestAll(test_keras):
        try:
                print("Test versions..")
                DisableTFWarns()
                Versions()
                TestGPU(0)
                if test_keras:
                        TestKeras()
                print("ALL OK")
        except Exception as ex:
                print(f"ERROR occured, caught exception {type(ex)}, '{ex}'")
                exit(-1) 

if __name__ == '__main__':
        parser = argparse.ArgumentParser()
        parser.add_argument("-k", default=False,action="store_true", help="run keras test")
        args = parser.parse_args()

        TestAll(test_keras=args.k)

usage: ipykernel_launcher.py [-h] [-k]
ipykernel_launcher.py: error: unrecognized arguments: -f /home/bram/.local/share/jupyter/runtime/kernel-280c09f5-f4b6-423d-96d1-a5a5b44681da.json


SystemExit: 2